
# Intelligent Asset Lifecycle Management
## Failure Prediction — Machine Learning Pipeline

This notebook follows the same overall architecture as the previous ML notebook:

**Load → Inspect → Clean → Feature Engineering → Train/Test Split → Preprocessing → Multiple ML Models → Compare → Best Model → Probability → Risk → Maintenance Recommendation**

### Dataset
Industrial machinery predictive-maintenance dataset (~24K rows).

### Prediction target
`failure_within_24h`

The model predicts whether a machine is likely to fail within the next 24 hours.

> **Important:** `rul_hours`, `failure_type`, and `estimated_repair_cost` are deliberately excluded from the failure model because they can leak information about the future/outcome. We will use them later for the lifecycle decision layer.


In [ ]:

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully.")


## 2. Load Dataset

In [ ]:

# Change this path if your CSV is stored somewhere else
DATA_PATH = "industrial_machine_predictive_maintenance.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()


## 3. Understand the Dataset

In [ ]:

print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())


In [ ]:

# Target distribution

print(df["failure_within_24h"].value_counts())
print("\nPercentage:")
print(df["failure_within_24h"].value_counts(normalize=True) * 100)

df["failure_within_24h"].value_counts().plot(
    kind="bar",
    title="Failure Within 24h Distribution"
)

plt.xlabel("Failure Within 24h")
plt.ylabel("Number of Records")
plt.show()



## 4. Basic Cleaning

We remove duplicate rows and the machine identifier.

We keep `machine_type` and `operating_mode` because they may contain useful information about different kinds of assets and operating conditions.


In [ ]:

df = df.drop_duplicates().reset_index(drop=True)

# Identifier: useful for tracking a machine, but not useful as an ML feature
if "machine_id" in df.columns:
    df = df.drop(columns=["machine_id"])

print("New shape:", df.shape)



## 5. Feature Engineering

We create a couple of physically meaningful features.

### Temperature rise
`motor temperature - ambient temperature`

This tells us how much additional heat the motor is generating relative to the environment.

### Electrical load index
`current × RPM`

This is a simple proxy for operating/electrical stress.

These are engineered features rather than new measurements.


In [ ]:

if {"temperature_motor", "ambient_temp"}.issubset(df.columns):
    df["temperature_rise"] = (
        df["temperature_motor"] - df["ambient_temp"]
    )

if {"current_phase_avg", "rpm"}.issubset(df.columns):
    df["electrical_load_index"] = (
        df["current_phase_avg"] * df["rpm"]
    )

df.head()



## 6. Select X and y

### Target

`failure_within_24h`

### Features excluded from the failure model

- `rul_hours` — future/remaining-life information
- `failure_type` — describes the failure outcome
- `estimated_repair_cost` — downstream information

Using these as predictors would make the model unrealistically powerful because it would have access to information that may only be known after/during the failure event.


In [ ]:

TARGET = "failure_within_24h"

leakage_columns = [
    TARGET,
    "rul_hours",
    "failure_type",
    "estimated_repair_cost"
]

X = df.drop(
    columns=[c for c in leakage_columns if c in df.columns]
)

y = df[TARGET].astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeatures:")
print(X.columns.tolist())


## 7. Train/Test Split

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))



## 8. Preprocessing

The dataset contains both numeric and categorical variables.

Numeric:
- temperature
- vibration
- pressure
- RPM
- etc.

Categorical:
- machine type
- operating mode

We:
1. Fill missing numeric values with the median.
2. Standardize numeric features.
3. Fill missing categorical values with the most frequent value.
4. One-hot encode categorical features.


In [ ]:

numeric_features = X_train.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)


In [ ]:

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features)
    ]
)

print("Preprocessor created.")


## 9. Define Machine Learning Models

In [ ]:

models = {

    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ),

    "SVM": SVC(
        probability=True,
        class_weight="balanced",
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=8,
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

print("Models:")
for name in models:
    print("-", name)


## 10. Train All Models and Compare Them

In [ ]:

results = []
trained_models = {}

for name, model in models.items():

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(
            y_test, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_test, y_pred, zero_division=0
        ),
        "F1 Score": f1_score(
            y_test, y_pred, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

    trained_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values(
    by="ROC-AUC",
    ascending=False
).reset_index(drop=True)

results_df



## 11. Select the Best Model

For this project, ROC-AUC is useful because the dataset is imbalanced: failures are less common than non-failures.

We will select the model with the highest ROC-AUC.


In [ ]:

best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

print("Best model:", best_model_name)


## 12. Detailed Evaluation

In [ ]:

best_pred = best_model.predict(X_test)
best_prob = best_model.predict_proba(X_test)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_test, best_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        best_pred,
        target_names=["No Failure", "Failure"],
        zero_division=0
    )
)

print("ROC-AUC:",
      roc_auc_score(y_test, best_prob))



## 13. Convert Failure Probability into an Asset Risk Level

The ML model gives us a probability.

We turn that into a human-readable asset risk level.

These thresholds are **business rules**, not learned by the ML model. They can later be tuned using the cost of downtime and maintenance.


In [ ]:

def risk_level(probability):

    if probability >= 0.75:
        return "CRITICAL FAILURE RISK"

    elif probability >= 0.50:
        return "HIGH FAILURE RISK"

    elif probability >= 0.25:
        return "MODERATE FAILURE RISK"

    else:
        return "LOW FAILURE RISK"


# Example
example_probability = 0.82

print("Probability:", example_probability * 100, "%")
print("Risk:", risk_level(example_probability))



## 14. Maintenance Decision Engine

This is the first step beyond simple predictive maintenance.

The ML model predicts risk.

The decision engine converts that prediction into an action:

- Monitor
- Plan maintenance
- Schedule maintenance soon
- Urgent maintenance

Later, we will extend this to:

**Repair vs Upgrade vs Replace**


In [ ]:

def maintenance_recommendation(
    failure_probability,
    hours_since_maintenance=None,
    rul_hours=None
):

    if failure_probability >= 0.75:
        return "URGENT MAINTENANCE"

    elif failure_probability >= 0.50:
        return "SCHEDULE MAINTENANCE SOON"

    elif rul_hours is not None and rul_hours < 100:
        return "PLAN MAINTENANCE"

    elif (
        hours_since_maintenance is not None
        and hours_since_maintenance > 500
    ):
        return "INSPECTION RECOMMENDED"

    else:
        return "MONITOR"


## 15. Test the System on One Asset

In [ ]:

# Pick one asset from the test set
sample = X_test.iloc[[0]]

probability = best_model.predict_proba(sample)[0][1]

original_row = df.loc[sample.index[0]]

hours_since_maintenance = original_row.get(
    "hours_since_maintenance",
    None
)

rul = original_row.get(
    "rul_hours",
    None
)

print("======================================")
print("       ASSET INTELLIGENCE REPORT")
print("======================================")

print("Model:", best_model_name)

print(
    "Failure Probability: {:.2f}%".format(
        probability * 100
    )
)

print(
    "Risk Level:",
    risk_level(probability)
)

print(
    "Recommendation:",
    maintenance_recommendation(
        probability,
        hours_since_maintenance,
        rul
    )
)

print(
    "RUL:",
    round(rul, 2),
    "hours"
)

print(
    "Hours Since Maintenance:",
    round(hours_since_maintenance, 2)
)



## 16. Predict Risk for Every Test Asset


In [ ]:

test_results = X_test.copy()

test_results["actual_failure"] = y_test.values

test_results["failure_probability"] = best_prob

test_results["risk_level"] = [
    risk_level(p)
    for p in best_prob
]

test_results["recommendation"] = [
    maintenance_recommendation(
        probability,
        df.loc[index, "hours_since_maintenance"],
        df.loc[index, "rul_hours"]
    )
    for probability, index in zip(
        best_prob,
        X_test.index
    )
]

test_results.head()


In [ ]:

print("Risk distribution:")
print(
    test_results["risk_level"].value_counts()
)

print("\nRecommendation distribution:")
print(
    test_results["recommendation"].value_counts()
)



# 17. Inspect High-Risk Assets

This is the type of output that can eventually become your hackathon dashboard.


In [ ]:

high_risk_assets = test_results[
    test_results["failure_probability"] >= 0.50
].sort_values(
    "failure_probability",
    ascending=False
)

high_risk_assets.head(20)



# 18. Save Predictions

The generated CSV can later be consumed by a Flask/FastAPI backend or dashboard.


In [ ]:

OUTPUT_PATH = "asset_failure_predictions.csv"

test_results.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Saved:", OUTPUT_PATH)



# Next Stage — Full Asset Lifecycle Intelligence

The failure model above is only the first layer.

Next we can build:

### Model 2 — Remaining Useful Life

**Input → sensor/operational data**

**Output → predicted RUL in hours**

### Lifecycle Decision Engine

Combine:

- Failure probability
- Predicted RUL
- Maintenance history
- Repair cost
- Asset criticality
- Utilization

Then produce:

**MONITOR / MAINTAIN / REPAIR / UPGRADE / REPLACE**

That will turn this from a predictive-maintenance project into a complete **Intelligent Asset Lifecycle Management** system.
